In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import json
from pathlib import Path
# import torch
# import lightning as L
# from utilities import CVDataModule
# from pytoda.smiles.smiles_language import SMILESTokenizer
# from paccmann_predictor.models import MODEL_FACTORY
# from cli_cv_paccmann import Module_training_paccmann

In [4]:
def precision_at_q(y_true, y_hat, q=0.25):
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    pred_topk = np.argsort(y_hat)[:k]

    return len(set(true_pos) & set(pred_topk)) / k

def ndcg_at_q(y_true, y_hat, q=0.25):
    "Normalized discounted cumulative gain"
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    # relevance: higher is better
    rel = -y_true
    
    # predicted ranking
    order = np.argsort(y_hat)
    rel_pred = rel[order][:k]
    
    discounts = 1 / np.log2(np.arange(2, k + 2)) # rank weight
    dcg = np.sum((2 ** rel_pred - 1) * discounts)

    # ideal ranking
    ideal_order = np.argsort(y_true)
    rel_ideal = rel[ideal_order][:k]
    idcg = np.sum((2 ** rel_ideal - 1) * discounts)

    return dcg / idcg if idcg > 0 else 0.0

In [7]:
params = {}
with open("paccmann_v2_params.json") as fp:
    params.update(json.load(fp))

params["number_of_genes"] = 2083
params["smiles_vocabulary_size"] = 87

In [ ]:
params

# Cross validation training

In [8]:
smiles_language = SMILESTokenizer.from_pretrained("smiles_language.pkl")
smiles_language.set_encoding_transforms(
            add_start_and_stop=params.get("add_start_and_stop", True),
            padding=params.get("padding", True),
            padding_length=params.get("smiles_padding_length", None),
        )
    
smiles_language.set_smiles_transforms(
            augment=params.get("augment_smiles", False),
            canonical=params.get("smiles_canonical", True),
            kekulize=params.get("smiles_kekulize", False),
            all_bonds_explicit=params.get("smiles_bonds_explicit", False),
            all_hs_explicit=params.get("smiles_all_hs_explicit", False),
            remove_bonddir=params.get("smiles_remove_bonddir", False),
            remove_chirality=params.get("smiles_remove_chirality", False),
            selfies=params.get("selfies", False),
            sanitize=params.get("selfies", False),
        )

In [4]:
model_name = params.get("model_fn", "paccmann_v2")
model = MODEL_FACTORY[model_name](params)
model._associate_language(smiles_language)


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_0/checkpoints/epoch=9-step=9670.ckpt


In [ ]:
root = "../../cross_validation"
cancer_type = "solid_tumors"
experiment = "NBS_cells"
ckpt_dir = sorted(Path(f"cv/models_{cancer_type}_NBS_cells/").glob("version_*"))
n=10

pred = {}

for f, ckpt in zip(range(n), ckpt_dir):
    ckpt_file = list((ckpt/"checkpoints").glob("*.ckpt"))[0]
    paccmann = Module_training_paccmann.load_from_checkpoint(
        ckpt_file,
        map_location="cuda",
        model=model,
        params = params
    )
    
    datamodule = CVDataModule(
            root = root,
            type = cancer_type,
            experiment = experiment,
            fold_n = f,
            GEX_path = "../../paccmann_GEX_data_filtered_logCPM.csv",
            SMILES_path = "../../drug_smiles.tsv",
            SMILES_language = smiles_language,
            batch_size = params["batch_size"],
            num_workers = 12
        )

    test_dl, test_set = datamodule.test_dataloader()

    trainer = L.Trainer(accelerator="auto")
    y_hat = trainer.predict(
        paccmann,
        dataloaders=test_dl
    )

    y_hat = torch.cat(y_hat).double().numpy()

    pred[f] = [y_hat]


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/solid_tumors_NBS_cells/version_0/checkpoints/epoch=9-step=8500.ckpt


/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_11/hparams.yaml
Predicting DataLoader 0: 100%|██████████████████████████████| 117/117 [00:06<00:00, 18.94it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/solid_tumors_NBS_cells/version_1/checkpoints/epoch=9-step=8470.ckpt


/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_12/hparams.yaml
Predicting DataLoader 0: 100%|██████████████████████████████| 120/120 [00:06<00:00, 18.57it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/solid_tumors_NBS_cells/version_2/checkpoints/epoch=9-step=8490.ckpt


...

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_18/hparams.yaml
Predicting DataLoader 0: 100%|██████████████████████████████| 117/117 [00:06<00:00, 18.63it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/solid_tumors_NBS_cells/version_8/checkpoints/epoch=9-step=8470.ckpt


/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_19/hparams.yaml
Predicting DataLoader 0: 100%|██████████████████████████████| 120/120 [00:06<00:00, 18.60it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/solid_tumors_NBS_cells/version_9/checkpoints/epoch=9-step=8470.ckpt


/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_20/hparams.yaml
Predicting DataLoader 0: 100%|██████████████████████████████| 120/120 [00:06<00:00, 18.53it/s]


In [ ]:
[pd.Series(v[0].flatten(), name=k).to_csv(f"CV/predictions_solid_tumors_NBS_cells/predictions/pred_fold_{k}.csv", index=False) for k,v in pred.items()]

[None, None, None, None, None, None, None, None, None, None]

# Cross validation evaluation

## NBS cell | Fixed-drug | Fixed-cell

In [6]:
root = Path("../../output/regression/cross_validation")
cancer_type = "pancancer"
experiment = "NBS_cells"
n=10

per_drug_pcc_cv = {}
per_cell_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}

for fold_name in range(n):
    train_set = pd.read_csv(
        root / cancer_type / experiment / f"fold_{fold_name}" / "train_set.csv",
        index_col=0
    )
    ic50_mean = train_set.LN_IC50.mean()
    ic50_std = train_set.LN_IC50.std()

    test_set = pd.read_csv(
        root / cancer_type / experiment / f"fold_{fold_name}" / "test_set.csv",
        index_col=0
    )
    print(root / cancer_type / experiment / f"fold_{fold_name}" / "test_set.csv")
    test_set["Y_TRUE"] = (test_set["LN_IC50"]-ic50_mean)/ic50_std

    y_hat = pd.read_csv(
        f"CV/predictions_{cancer_type}_{experiment}/pred_fold_{fold_name}.csv"
    )

    test_set["Y_HAT"] = y_hat.to_numpy()

    # Computing metrics
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_cell_pcc_cv[fold_name] = [per_cell_pcc["PCC"].median()]

    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]




../../output/regression/cross_validation/pancancer/NBS_cells/fold_0/test_set.csv


/tmp/ipykernel_26180/2360469460.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or expli

../../output/regression/cross_validation/pancancer/NBS_cells/fold_1/test_set.csv


/tmp/ipykernel_26180/2360469460.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or expli

../../output/regression/cross_validation/pancancer/NBS_cells/fold_2/test_set.csv


...

/tmp/ipykernel_26180/2360469460.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or expli

../../output/regression/cross_validation/pancancer/NBS_cells/fold_8/test_set.csv


/tmp/ipykernel_26180/2360469460.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or expli

../../output/regression/cross_validation/pancancer/NBS_cells/fold_9/test_set.csv


/tmp/ipykernel_26180/2360469460.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_26180/2360469460.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or expli

In [7]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["Paccmann"]
per_drug_pcc_cv

,0,1,2,3,4,5,6,7,8,9
Paccmann,0.277194,0.288726,0.445097,0.516187,0.232899,0.45059,0.347161,0.510072,0.525021,0.256598


In [8]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.3961288596858338, low=0.3127329134946504, high=0.4795248058770172


In [ ]:
# per_drug_pcc_cv.to_csv(
#     "CV/predictions_solid_tumors_NBS_cells/fixed-drug_evaluation_df.csv"
# )

In [9]:
per_cell_pcc_cv = pd.DataFrame(per_cell_pcc_cv)
per_cell_pcc_cv.index = ["Paccmann"]
per_cell_pcc_cv

,0,1,2,3,4,5,6,7,8,9
Paccmann,0.705541,0.69587,0.688858,0.739093,0.413669,0.695262,0.710054,0.704452,0.724838,0.700232


In [10]:
M = np.median(per_cell_pcc_cv)
sem = stats.sem(per_cell_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.7023422286918523, low=0.6351039504041812, high=0.7695805069795233


In [ ]:
# per_cell_pcc_cv.to_csv(
#     "CV/predictions_solid_tumors_NBS_cells/fixed-cell_evaluation_df.csv"
# )

In [11]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["Paccmann"]
per_drug_precision_cv

,0,1,2,3,4,5,6,7,8,9
Paccmann,0.478261,0.444444,0.521739,0.529412,0.478261,0.5,0.454545,0.555556,0.521739,0.368421


In [15]:
per_drug_precision_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/precision_fixed-drug_CV.csv"
)

In [12]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["Paccmann"]
per_drug_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
Paccmann,0.268795,0.277148,0.41731,0.442117,0.344707,0.301362,0.328657,0.458292,0.360554,0.209433


In [16]:
per_drug_ndcg_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/ndcg_fixed-drug_CV.csv"
)

In [13]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["Paccmann"]
per_cell_precision_cv

,0,1,2,3,4,5,6,7,8,9
Paccmann,0.742929,0.729167,0.745098,0.726316,0.680412,0.732673,0.723404,0.714286,0.708333,0.72619


In [17]:
per_cell_precision_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/precision_fixed-cell_CV.csv"
)

In [14]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["Paccmann"]
per_cell_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
Paccmann,0.695153,0.679004,0.705435,0.670185,0.534964,0.658366,0.680307,0.647557,0.665716,0.651756


In [18]:
per_cell_ndcg_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv"
)

## NBS cell | Fixed-pathway | Fixed-TCGA

In [3]:
root = Path("../../output/regression/cross_validation")
cancer_type = "pancancer"
experiment = "NBS_cells"
n=10

per_path_pcc_cv = {}
per_tcga_pcc_cv = {}

for fold_name in range(n):
    train_set = pd.read_csv(
        root / cancer_type / experiment / f"fold_{fold_name}" / "train_set.csv",
        index_col=0
    )
    ic50_mean = train_set.LN_IC50.mean()
    ic50_std = train_set.LN_IC50.std()

    test_set = pd.read_csv(
        root / cancer_type / experiment / f"fold_{fold_name}" / "test_set.csv",
        index_col=0
    )
    print(root / cancer_type / experiment / f"fold_{fold_name}" / "test_set.csv")
    test_set["Y_TRUE"] = (test_set["LN_IC50"]-ic50_mean)/ic50_std

    y_hat = pd.read_csv(
        f"CV/predictions_{cancer_type}_{experiment}/pred_fold_{fold_name}.csv"
    )

    test_set["Y_HAT"] = y_hat.to_numpy()

    pathway = (test_set[["DRUG_NAME","PATHWAY_NAME"]]
             .drop_duplicates())
    
    tcga_desc = (test_set[["CELL_LINE_NAME","TCGA_DESC"]]
             .drop_duplicates())

    # Computing metrics
    # Computing metrics
    per_path_pcc = (test_set
                    .groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_path_pcc = (per_path_pcc
                    .merge(pathway, on="DRUG_NAME", how="left")
                    .dropna(subset=["PATHWAY_NAME"])
                    .groupby("PATHWAY_NAME")["PCC"]
                    .median()
                    )

    per_path_pcc_cv[fold_name] = per_path_pcc

    per_tcga_pcc = (test_set
                    .groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    
    per_tcga_pcc = (per_tcga_pcc.
                    merge(tcga_desc, on="CELL_LINE_NAME", how="left").
                    dropna(subset="TCGA_DESC")
                    .groupby("TCGA_DESC")["PCC"]
                    .median()
                    )

    per_tcga_pcc_cv[fold_name] = per_tcga_pcc


../../output/regression/cross_validation/pancancer/NBS_cells/fold_0/test_set.csv


/tmp/ipykernel_68931/2672508693.py:42: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the grouping

../../output/regression/cross_validation/pancancer/NBS_cells/fold_1/test_set.csv


/tmp/ipykernel_68931/2672508693.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


../../output/regression/cross_validation/pancancer/NBS_cells/fold_2/test_set.csv


...

/tmp/ipykernel_68931/2672508693.py:42: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the grouping

../../output/regression/cross_validation/pancancer/NBS_cells/fold_8/test_set.csv


/tmp/ipykernel_68931/2672508693.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


../../output/regression/cross_validation/pancancer/NBS_cells/fold_9/test_set.csv


/tmp/ipykernel_68931/2672508693.py:42: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_68931/2672508693.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the grouping

In [4]:
exclude = ["Unclassified", "Other", "Other, kinases", "Chromatin other"]

In [6]:
path = (pd.DataFrame(per_path_pcc_cv)
 .groupby(level=0)
 .median()
 .drop(exclude)
#  .median(axis=1)
 .rename_axis("")
 .T
 )

path["fold"] = path.index

path = (path
        .melt(id_vars="fold", var_name="PATHWAY_NAME", value_name="MCorrelation")
        )

path = path.assign(model="Paccmann")
path.to_csv(
    "CV/predictions_pancancer_NBS_cells/fixed-drug_PATHWAY_CV.csv"
)

In [7]:
tcga = (pd.DataFrame(per_tcga_pcc_cv)
        .dropna()
        # .median(axis=1)
        .drop(["UNCLASSIFIED"])
        .rename_axis("")
 )

tcga = pd.DataFrame(tcga).T

tcga["fold"] = tcga.index

tcga = (tcga
        .melt(id_vars = "fold", var_name="TCGA_DESC", value_name="MCorrelation")
        )


tcga = tcga.assign(model="Paccmann")
tcga.to_csv("CV/predictions_pancancer_NBS_cells/fixed-cell_TCGA_CV.csv")
tcga

,fold,TCGA_DESC,MCorrelation,model
0,0,ALL,0.431419,Paccmann
1,1,ALL,0.632899,Paccmann
2,2,ALL,0.551583,Paccmann
3,3,ALL,0.589153,Paccmann
4,4,ALL,0.292134,Paccmann
...,...,...,...,...
115,5,SKCM,0.712791,Paccmann
116,6,SKCM,0.706173,Paccmann
117,7,SKCM,0.740242,Paccmann
118,8,SKCM,0.737810,Paccmann
